# ANALISIS DEL MODELO ARIMA - KAKEBIA

Esta celda instala la librería `statsmodels` en el entorno de trabajo, en caso de que no se encuentre disponible. Se trata de una biblioteca de Python especializada en análisis estadístico, la cual proporciona la implementación del modelo **ARIMA** empleado para el pronóstico de la serie temporal de gastos. Únicamente es necesario ejecutarla una vez por entorno de trabajo.


In [1]:
#!pip install statsmodels

# \-\-\-\-\-\-\-\-\-\- Importar librerías \-\-\-\-\-\-\-\-\-\-

Esta celda importa todas las librerías necesarias para el desarrollo del análisis:

- `os` y `shutil`: módulos estándar de Python utilizados para operaciones sobre el sistema de archivos, como copiar y mover archivos entre directorios.
- `pandas` (`pd`): biblioteca para la carga, la manipulación y la transformación de datos tabulares.
- `numpy` (`np`): biblioteca para operaciones numéricas y manejo de valores faltantes (`NaN`).
- `ARIMA` de `statsmodels.tsa.arima.model`: clase que implementa el modelo ARIMA para el análisis de series temporales.
- `plotly.graph_objects` (`go`): módulo empleado para construir gráficas interactivas.

In [ ]:
import os
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
import plotly.graph_objects as go
import shutil

## \-\-\-\-\-\-\-\-\-\- 1\) Cargar y limpiar los datos\-\-\-\-\-\-\-\-\-\-

## Cargar datos

Esta celda define la ruta del archivo CSV de datos procesados y lo carga en un `DataFrame` de pandas. Se emplea una instrucción condicional `if` para verificar que la ruta corresponda al valor esperado antes de intentar la lectura del archivo; de lo contrario, se lanza un `ValueError` con un mensaje de error descriptivo que facilita la identificación del problema.

In [3]:
path = "../data/processed/kakebo_merged.csv"

# Condicional 'if' para leer archivo .csv

if path == "../data/processed/kakebo_merged.csv":
    df = pd.read_csv(path, sep=";", encoding="utf-8")
    print(f"Nota: El archivo '{path}' fue cargado exitosamente.")
else:
    raise ValueError(f"Error: El archivo '{path}' no fue cargado exitosamente." )

Nota: El archivo '../data/processed/kakebo_merged.csv' fue cargado exitosamente.


## Limpiar y parsear datos

Esta celda realiza la limpieza y la preparación de los datos para el modelo, siguiendo los pasos descritos a continuación:

1. **Estandarización de columnas**: convierte los nombres de todas las columnas a mayúsculas y elimina espacios adicionales al inicio o al final.
2. **Conversión del monto**: la función `parse_monto` suprime los separadores de miles (`.`) y convierte la coma decimal en punto, transformando el valor de texto a tipo `float`.
3. **Mapeo de meses**: extrae el nombre del mes del campo `MES`, lo asigna a su número correspondiente (del 1 al 12) y construye la columna `FECHA` con formato `datetime`.
4. **Filtrado de datos reales**: si existe la columna `TIPO_DATO`, se utilizan únicamente las filas etiquetadas como `'REAL'` para entrenar el modelo, evitando la contaminación con predicciones anteriores.
5. **Construcción de la serie temporal**: convierte el DataFrame filtrado en una serie indexada por fecha, con frecuencia mensual (`MS`).

In [4]:
df.columns = [c.strip().upper() for c in df.columns]

def parse_monto(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = s.replace(".", "")      # miles
    s = s.replace(",", ".")     # decimal
    return float(s)

df["MONTO"] = df["MONTO"].apply(parse_monto)

df["MES_NOMBRE"] = df["MES"].astype(str).str.replace("MES", "", regex=False).str.strip().str.upper()

mes_map = {
    "ENERO": 1, "FEBRERO": 2, "MARZO": 3, "ABRIL": 4, "MAYO": 5, "JUNIO": 6,
    "JULIO": 7, "AGOSTO": 8, "SEPTIEMBRE": 9, "SETIEMBRE": 9, "OCTUBRE": 10,
    "NOVIEMBRE": 11, "DICIEMBRE": 12
}
df["MES_NUM"] = df["MES_NOMBRE"].map(mes_map)
df["FECHA"] = pd.to_datetime(dict(year=df["AÑO"], month=df["MES_NUM"], day=1))
df = df.sort_values("FECHA")

# Entrenar SOLO con datos reales
if "TIPO_DATO" in df.columns:
    train_df = df[df["TIPO_DATO"].astype(str).str.upper() == "REAL"].copy()
    if train_df.empty:
        train_df = df.copy()
else:
    train_df = df.copy()
y = train_df.set_index("FECHA")["MONTO"].asfreq("MS")

## \-\-\-\-\-\-\-\-\-\- 2\) Implementar y entrenar modelo ARIMA \-\-\-\-\-\-\-\-\-\-

Esta celda define los hiperparámetros del modelo ARIMA mediante la tupla `order = (p, d, q)`, cuyos componentes se describen a continuación:

- **p = 1**: número de términos autorregresivos (AR); indica que el modelo considera el valor del período inmediatamente anterior.
- **d = 1**: orden de diferenciación; se aplica una diferencia a la serie para hacerla estacionaria.
- **q = 1**: número de términos de media móvil (MA); el modelo corrige su estimación a partir del error del período anterior.

Los parámetros `enforce_stationarity=False` y `enforce_invertibility=False` otorgan mayor flexibilidad durante el ajuste. Por último, `model.fit()` estima los coeficientes del modelo con los datos de entrenamiento y retorna el objeto de resultados `res`.

In [5]:
order = (1, 1, 1)
model = ARIMA(y, order=order, enforce_stationarity=False, enforce_invertibility=False)
res = model.fit()

## \-\-\-\-\-\-\-\-\-\- 3\) Pronóstico a 12 meses \-\-\-\-\-\-\-\-\-\-

Esta celda genera el pronóstico para los **12 meses** siguientes, utilizando el modelo ARIMA previamente ajustado:

- `get_forecast(steps=12)`: calcula las predicciones para los próximos 12 períodos mensuales.
- `predicted_mean`: contiene el valor medio de cada predicción puntual, es decir, el gasto mensual esperado.
- `conf_int(alpha=0.05)`: devuelve el **intervalo de confianza al 95 %**, compuesto por los límites inferior y superior dentro de los cuales se estima, con un 95 % de probabilidad, que se encontrará el valor real.

In [6]:
steps = 3
forecast_res = res.get_forecast(steps=steps)

yhat = forecast_res.predicted_mean #uso de la media para hacer las predicciones
ci = forecast_res.conf_int(alpha=0.05)  # uso del 95% en el rango Intervalo de Confianza

## \-\-\-\-\-\-\-\-\-\- 4\) Predicción con intervalo de confianza IC\-\-\-\-\-\-\-\-\-\-

Esta celda construye el `DataFrame` de salida que integra los datos históricos y las predicciones:

1. **Histórico (`hist_out`)**: toma la serie `y` con los datos reales, asigna la etiqueta `'REAL'` en la columna `tipo_dato` e introduce valores `NaN` en los límites del intervalo de confianza, dado que no aplican para registros históricos.
2. **Pronóstico (`pred_out`)**: construye un `DataFrame` con las fechas futuras, los valores predichos, la etiqueta `'PREDICCION'` y los límites inferior (`lower_95`) y superior (`upper_95`) del intervalo de confianza.
3. **Unión y formato**: concatena ambos DataFrames ordenados cronológicamente por fecha y aplica el formato estándar `YYYY-MM-DD` a la columna `fecha`.

In [7]:
# Histórico
hist_out = (
    y.reset_index()
     .rename(columns={"FECHA": "fecha", "MONTO": "monto"})
)
hist_out["tipo_dato"] = "REAL"
hist_out["lower_95"] = np.nan
hist_out["upper_95"] = np.nan

# Pronóstico
pred_out = pd.DataFrame({
    "fecha": yhat.index,
    "monto": yhat.values,
    "tipo_dato": "PREDICCION",
    "lower_95": ci.iloc[:, 0].values,
    "upper_95": ci.iloc[:, 1].values
})

out = pd.concat([hist_out, pred_out], ignore_index=True).sort_values("fecha")

# Opcional: formatear fecha YYYY-MM-DD
out["fecha"] = out["fecha"].dt.strftime("%Y-%m-%d")

### \-\-\-\-\-\-\-\-\-\- 4\.1\) Exportar histórico \+ predicción \-\-\-\-\-\-\-\-\-\-

Esta celda exporta el `DataFrame` combinado —que contiene tanto el histórico como las predicciones— al archivo `kakebo_pred_hist.csv`, ubicado en la carpeta de datos procesados. A continuación, imprime una confirmación de la exportación y muestra las últimas 15 filas del resultado, lo que permite verificar rápidamente la integridad del contenido generado.

In [8]:
out_path = "../data/processed/kakebo_pred_hist.csv"
out.to_csv(out_path, index=False, encoding="utf-8")

print(f"Archivo exportado: {out_path}")
print(out.tail(15).to_string(index=False))

Archivo exportado: ../data/processed/kakebo_pred_hist.csv
     fecha        monto  tipo_dato     lower_95     upper_95
2025-09-01 5.206932e+06       REAL          NaN          NaN
2025-10-01 6.291127e+06       REAL          NaN          NaN
2025-11-01 5.725800e+06       REAL          NaN          NaN
2025-12-01 6.068471e+06       REAL          NaN          NaN
2026-01-01 8.705436e+06       REAL          NaN          NaN
2026-02-01 5.916236e+06       REAL          NaN          NaN
2026-03-01 5.092541e+06       REAL          NaN          NaN
2026-04-01 5.883259e+06       REAL          NaN          NaN
2026-05-01 7.414040e+06       REAL          NaN          NaN
2026-06-01 6.040126e+06       REAL          NaN          NaN
2026-07-01 5.631825e+06       REAL          NaN          NaN
2026-08-01 4.696388e+06       REAL          NaN          NaN
2026-09-01 4.931447e+06 PREDICCION 2.550190e+06 7.312705e+06
2026-10-01 4.946088e+06 PREDICCION 1.912386e+06 7.979790e+06
2026-11-01 4.947000e+06 PRE

## \-\-\-\-\-\-\-\-\-\- Gráficas de serie temporal \-\-\-\-\-\-\-\-\-\-

Esta celda crea un **gráfico interactivo** con Plotly para visualizar la serie temporal completa, compuesto por los siguientes elementos:

1. **Datos históricos reales**: traza de línea con marcadores que representa los gastos registrados.
2. **Línea de predicción**: traza que muestra los valores pronosticados para los próximos 12 meses.
3. **Banda del intervalo de confianza (IC 95 %)**: franja sombreada construida con dos trazas superpuestas (`fill='tonexty'`), que delimita los límites inferior y superior del pronóstico e indica visualmente el grado de incertidumbre del modelo.

El diseño del gráfico incluye el título centrado, las etiquetas de los ejes, el tema visual `plotly_white` y el modo `hovermode='x unified'`, que permite visualizar simultáneamente todos los valores al desplazar el cursor sobre una fecha.

In [9]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=pd.to_datetime(hist_out["fecha"]),
    y=hist_out["monto"].values,
    mode="lines+markers",
    name="Datos Históricos (REALES)"
))

fig.add_trace(go.Scatter(
    x=yhat.index, y=yhat.values,
    mode="lines+markers",
    name="Predicción"
))

fig.add_trace(go.Scatter(
    x=ci.index,
    y=ci.iloc[:, 0].values,
    mode="lines",
    line=dict(width=0),
    showlegend=False
))
fig.add_trace(go.Scatter(
    x=ci.index,
    y=ci.iloc[:, 1].values,
    mode="lines",
    fill="tonexty",
    line=dict(width=0),
    name="Rango de Predicción - (IC 95%)"
))

fig.update_layout(
    title=f"Predicciones de gastos KAKEBO ",
    title_x=0.5,
    title_xanchor="center",
    xaxis_title="Fecha",
    yaxis_title="Monto",
    template="plotly_white",
    hovermode="x unified"
)
fig.show()

Se habla de que el Intervalo de confianza, es el rango que estima dónde podría estar ese valor verdadero dentro de la franja morada, el punto rojo, es el valor de la predicción estimada a gastar durante los siguientes meses\.

# \-\-\-\-\-\-\-\-\-\- Exportar para el tablero de PowerBI \-\-\-\-\-\-\-\-\-\-

Esta celda prepara y exporta los datos en el formato requerido para el tablero de **Power BI**, siguiendo los pasos indicados a continuación:

1. Lee el archivo CSV generado anteriormente (`kakebo_pred_hist.csv`).
2. Elimina las columnas del intervalo de confianza (`lower_95` y `upper_95`), dado que no son necesarias en el tablero visual.
3. Renombra las columnas con letras mayúsculas para mantener la uniformidad del esquema de datos.
4. Redondea el campo `MONTO` al entero más cercano (tipo `Int64`), evitando decimales en el tablero.
5. La función `formato_pesos_colombianos` convierte el monto numérico a texto con el símbolo `$` y los separadores de miles correspondientes; por ejemplo: `$ 1.500.000`.
6. Exporta el resultado al archivo `kakebo_pred_pbix.csv`, listo para ser importado en Power BI.

In [10]:
df = pd.read_csv("../data/processed/kakebo_pred_hist.csv")

df = df.drop(columns=["lower_95", "upper_95"], errors="ignore").rename(columns={
    "fecha": "FECHA",
    "monto": "MONTO",
    "tipo_dato": "TIPO_DATO",
})

df["MONTO"] = pd.to_numeric(df["MONTO"], errors="coerce").round(0).astype("Int64")

# sin la palabra COP
def formato_pesos_colombianos(x):
    if pd.isna(x):
        return None
    return f"$ {int(x):,}".replace(",", ".")

df["MONTO_COP"] = df["MONTO"].apply(formato_pesos_colombianos)



df.to_csv("../data/processed/kakebo_pred_pbix.csv", index=False)
print(df[["FECHA", "MONTO_COP", "TIPO_DATO"]].head())

        FECHA     MONTO_COP TIPO_DATO
0  2025-01-01  $ 14.057.968      REAL
1  2025-02-01   $ 3.310.580      REAL
2  2025-03-01   $ 3.072.536      REAL
3  2025-04-01   $ 3.455.498      REAL
4  2025-05-01   $ 3.385.952      REAL


# \-\-\-\-\-\-\-\-\-\- Exportar gráfico a formato HTML \-\-\-\-\-\-\-\-\-\-

Esta celda exporta el gráfico interactivo de Plotly como un archivo **HTML autónomo** (`arima_forecast.html`) en la carpeta `dashboards`, lo que permite visualizarlo en cualquier navegador web sin necesidad de contar con Python instalado. Adicionalmente, copia dicho archivo al directorio del módulo de dashboard del sistema **REDOHIS** en el servidor WINDOWS SERVER casero, dejando el gráfico disponible directamente en la aplicación web.

In [ ]:

fig.write_html("../dashboards/arima_forecast.html")
print("Gráfico exportado a: arima_forecast.html")


#exportar informe a la carpeta del dashboard de REDOHIS
output_path = r"C:\xampp\htdocs\REDOHIS\modules\dashboard\arima_forecast.html"
shutil.copy("../dashboards/arima_forecast.html", output_path)
print(f"Gráfico también exportado a: {output_path}")

Gráfico exportado a: arima_forecast.html
Gráfico también exportado a: C:\xampp\htdocs\REDOHIS\modules\dashboard\arima_forecast.html
